# AIS Positions取得

通常は「設定」セルだけ編集し、上から順に実行します。

実行中はブラウザと進捗バーを確認できます。

In [ ]:
import os
from pathlib import Path
import sys
import importlib
import polars as pl

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "tools").is_dir() and (path / "vessel").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from tools.scrapers import ais_positions as sss_ais

sss_ais = importlib.reload(sss_ais)


In [ ]:
# ===== 設定：通常はここだけ編集 =====
# PERIOD: 取得期間。"all"で全期間、またはfrom/toにYYYY-MM-DD形式の日付を指定します。
PERIOD = {"from": "2008-01-01", "to": "2025-12-31"}

# ACCOUNT_ID: このノートブックが使用するcoordinator上のアカウントです。
ACCOUNT_ID = "account_1"

# LOGIN_CONFIG: アカウント1専用のSeaSearcherログイン情報を読み込みます。
LOGIN_CONFIG = {
    "login_user": os.getenv("SEASEARCHER_LOGIN_USER_1"),
    "login_password": os.getenv("SEASEARCHER_LOGIN_PASSWORD_1"),
}
if not all(LOGIN_CONFIG.values()):
    raise ValueError("SEASEARCHER_LOGIN_USER_1 と SEASEARCHER_LOGIN_PASSWORD_1 を .env または環境変数に設定してください。")

# SOURCE_CONFIG: 入力元CSVの読み込み方法を指定します。
# source_mode: "single_file"=任意の1ファイル、"split_files"=船種別CSV群です。
# source_vessel_file: single_fileで使用するCSV。split_filesではNoneで構いません。
# source_statuses: "all"=全Status、またはLive / Unconfirmed Existence / Deadのリストで絞り込みます。
# dir_vessel: split_filesで船種別CSVを探すフォルダです。
# vessel_type_list: split_filesで読み込む船種のリストです。
# file_template: split_filesのファイル名テンプレート。{vessel_type}が船種名に置換されます。
SOURCE_CONFIG = {
    "source_mode": "single_file",  # 入力モード: "single_file" / "split_files"
    "source_vessel_file": r"vessel/vessels_tokyo_LNG.csv",  # single_fileで使うCSV
    "source_statuses": "all",  # 対象Status: "all"、またはStatus名のリスト
    "dir_vessel": "vessel",  # split_filesでCSVを探すフォルダ
    "vessel_type_list": ["bulk", "container", "cruise", "generalcargo", "roro", "tanker", "vehicle"],  # split_filesの船種
    "file_template": "vessels_202504_{vessel_type}.csv",  # split_filesのファイル名テンプレート
}

# RUN_CONFIG: 対象船の範囲と出力先を指定します。
# run_vessel_type: "all"、またはCSV内の"LLI Vessel Type"の値で船種を絞り込みます。
# run_start/run_end: 対象リストのスライス範囲。全件は0とNoneです。
# reverse_targets: True=対象リストを逆順にしてから範囲を適用、False=CSVの並び順です。
# max_workers: AISでは安全性とサイト負荷のため通常1を指定します。utility側でも1に固定されます。
# out_dir: 最終AIS CSVの出力先。Noneなら入力元と船種から自動決定します。
RUN_CONFIG = {
    "run_vessel_type": "all",  # 対象船種: "all"、またはCSV内のLLI Vessel Type
    "run_start": 0,  # 対象リストの開始位置（0始まり）
    "run_end": None,  # 対象リストの終了位置（None=末尾まで）
    "reverse_targets": False,  # True=逆順、False=入力順。アカウント1は入力順
    "max_workers": 1,  # 同時実行数。AISでは実質1に固定
    "out_dir": "LNG_2008_2025",  # AIS CSVの出力先。None=自動設定
}

# SCRAPING_CONFIG: AIS取得時の実用設定です。
# period: 取得期間。"all"、またはfrom/toの辞書です。
# local_time: False=GMT、True=SeaSearcher画面のLocal Timeです。
# headless: False=Chromeを表示、True=非表示です。初回はFalseを推奨します。
# log_level: "DONE"=完了中心、"INFO"=通常、"DEBUG"=詳細ログです。
# check_status: Trueなら対象船のStatus確認を追加します。
# skip_if_exists: 既存の同期間AIS CSVがあれば取得をスキップします。
# skip_if_known_no_data: no-data記録済みの船をスキップします。
# remember_no_data: データなしの結果を次回用の記録に保存します。
# show_progress: Trueなら全体・船ごとの進捗を表示します。
# encoding: 出力CSVの文字コードです。
# log_steps: Trueならページ操作など各ステップのログを表示します。
# periodic_rest_enabled: Trueなら連続稼働後に休止を入れます。
# work_session_hours / work_session_random_minutes: 休止までの基準稼働時間とランダム幅（分）です。
# rest_session_hours / rest_session_random_minutes: 休止の基準時間とランダム幅（分）です。
# retry_timeout_immediately: タイムアウト直後の再試行。通常はFalseです。
# relogin_on_confirmed_session_loss: ログイン画面への遷移を確認したとき再ログインします。
# session_relogin_attempts: 1回の認証切れに対する再ログイン回数です。
# stop_on_access_block: アクセス拒否・レート制限・CAPTCHA検知時に全体を停止します。
# webdriver_restart_attempts: WebDriver再起動の追加試行回数です。
# failed_llino_retry_max_attempts: 船単位の失敗を待機後に再試行する最大回数です。
# period_window_retry_attempts: 期間ウィンドウ単位の再試行回数です。
# between_period_windows_delay_seconds: 期間ウィンドウ間の待機秒数の範囲です。
# max_consecutive_failed_llinos: 連続失敗がこの数に達したときの停止判定に使います。
SCRAPING_CONFIG = {
    "coordinator_account_id": ACCOUNT_ID,  # coordinatorで明示的にaccount_1を使用
    "period": PERIOD,  # 取得期間: "all"、またはfrom/toの辞書
    "local_time": False,  # False=GMT、True=Local Time
    "out_dir": RUN_CONFIG["out_dir"],  # 出力先。確認セルで解決済みパスに更新
    "headless": False,  # False=ブラウザ表示、True=非表示
    "login_user": LOGIN_CONFIG["login_user"],  # LOGIN_CONFIGで指定したログインユーザー
    "login_password": LOGIN_CONFIG["login_password"],  # LOGIN_CONFIGで指定したパスワード
    "log_level": "DONE",  # ログレベル: DONE / INFO / DEBUG
    "log_steps": False,  # True=各処理ステップをログ出力
    "periodic_rest_enabled": True,  # True=定期休止を有効化
    "work_session_hours": 6,  # 休止までの基準稼働時間
    "work_session_random_minutes": 30,  # 稼働時間に加えるランダム幅（分）
    "rest_session_hours": 1,  # 休止の基準時間
    "rest_session_random_minutes": 15,  # 休止時間に加えるランダム幅（分）
    "retry_timeout_immediately": False,  # True=タイムアウト直後に再試行
    "relogin_on_confirmed_session_loss": True,  # True=認証切れ確認時に再ログイン
    "session_relogin_attempts": 1,  # 認証切れ1回あたりの再ログイン上限
    "stop_on_access_block": True,  # True=アクセス拒否等を検知したら全体停止
    "webdriver_restart_attempts": 0,  # WebDriver再起動の追加試行回数
    "failed_llino_retry_max_attempts": 2,  # 船単位の失敗の再試行上限
    "period_window_retry_attempts": 1,  # 期間ウィンドウの再試行回数
    "between_period_windows_delay_seconds": (2.0, 5.0),  # 期間ウィンドウ間の待機秒数範囲
    "max_consecutive_failed_llinos": 3,  # 連続失敗時の停止判定数
    "check_status": False,  # True=対象船のStatusを確認
    "skip_if_exists": True,  # True=既存の同期間CSVをスキップ
    "skip_if_known_no_data": True,  # True=no-data記録済みの船をスキップ
    "remember_no_data": True,  # True=no-data結果を記録
    "show_progress": True,  # True=進捗を表示
    "encoding": "utf-8-sig",  # 出力CSVの文字コード
}

LOGIN_CONFIG, SOURCE_CONFIG, RUN_CONFIG, SCRAPING_CONFIG


In [ ]:
# 対象船・既存出力・出力先を確認します。ここではサイトへ接続しません。
# reverse_targets後にrun_start/run_endを適用し、movement版と同じ順序にします。
run_context = sss_ais.prepare_ais_run_context(
    source_config=SOURCE_CONFIG,  # 入力CSVの読み込み設定
    run_vessel_type=RUN_CONFIG["run_vessel_type"],  # 対象船種
    run_start=0,  # ここでは全候補を読み込み、下で範囲を適用
    run_end=None,  # ここでは全候補を読み込み、下で範囲を適用
    out_dir=RUN_CONFIG["out_dir"],  # 出力先。Noneなら自動設定
)
ordered_targets = (
    list(reversed(run_context["targets"]))
    if RUN_CONFIG.get("reverse_targets", False)
    else list(run_context["targets"])
)
targets_to_run = ordered_targets[RUN_CONFIG["run_start"]:RUN_CONFIG["run_end"]]
SCRAPING_CONFIG["period"] = PERIOD
SCRAPING_CONFIG["out_dir"] = run_context["out_dir"]
preview_config = {**SCRAPING_CONFIG}
existing_output_count = sum(
    bool(sss_ais._has_saved_ais_positions_for_llino(llino, preview_config))
    for llino in targets_to_run
)
target_summary = {
    "source_description": run_context["source_description"],
    "total_candidates": len(run_context["targets"]),
    "target_count_to_scrape": len(targets_to_run),
    "existing_output_count": existing_output_count,
    "run_start": RUN_CONFIG["run_start"],
    "run_end": RUN_CONFIG["run_end"],
    "reverse_targets": RUN_CONFIG["reverse_targets"],
    "run_vessel_type": RUN_CONFIG["run_vessel_type"],
    "period": PERIOD,
    "out_dir": SCRAPING_CONFIG["out_dir"],
    "first_10_llino": targets_to_run[:10],
}
target_summary


In [ ]:
# AIS Positionsをスクレイピングします。ここで初めてChromeが開き、ログインします。
# targets_to_run: 前セルで確認した対象LLIのリストです。
# max_workers: 同時実行数です。AISではutility側でも1に固定されます。
# config: 期間・出力先・ログ・再試行などの実行設定です。
results = sss_ais.parallel_scraping_ais_positions(
    targets_to_run,
    max_workers=RUN_CONFIG["max_workers"],
    config=SCRAPING_CONFIG,
)

ok_count = sum(bool(result.get("ok")) for result in results)
failed_count = len(results) - ok_count
{"total": len(results), "ok": ok_count, "failed": failed_count}
